[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/bending/bending.ipynb)

# Bending and Twisting of Thin-Walled Beams

This example computes how thin-walled beams bend, slide, and twist under end loads in Euler–Bernoulli beam theory with Timoshenko shear and Saint-Venant torsion. Each beam is modelled as a bundle of axial fibres joined into thin straight walls whose cross-sections stay planar under bending and warp axially under torsion. Under bending, neighbouring cross-sections turn slightly relative to one another about a neutral axis in the section plane, stretching the fibres on one side and compressing those on the other, while open thin walls combine high resistance to bending and midline shear with low resistance to twisting about the beam. Asymmetrical sections deflect sideways when loaded off their principal bending directions, and open sections twist whenever the load line misses their shear centre.

In the projective geometric algebra of three-dimensional space, strains and loads are both lines of opposite orientation, represented by bivector twists and antibivector forques. Joining a cross-section point with `beam_direction` forms the axial fibre `beam_direction & point`, whose elongation rate under a strain is the regressive pairing `strain & fibre`. Summing line dyads for fibre tension `modulus * fibres * (Twist & fibres)`, Saint-Venant torsion, and wall shear through the shear centre builds one section stiffness map without precomputing a centroid or lever arms. Because every wall's shear line passes through the shear centre, twisting about the axial line through that centre produces zero wall slide, so `stiffness.solve(-load)` places the twisting axis there automatically and motor exponentials integrate the piece twists into cross-section frames (wall quadrature follows the [simplex inertia example](../simplex.py)).

In matrix beam-theory notation, a cross-section is first shifted to its centroid $(x_c, y_c)$ and shear centre $(x_s, y_s)$ after evaluating the first moments $\int x\,dA, \int y\,dA$, second moments $I_{xx}, I_{yy}, I_{xy}$, and sectorial warping moments $\int \omega x\,dA, \int \omega y\,dA$, before assembling a $6 \times 6$ constitutive matrix mapping $(\varepsilon_z, \gamma_x, \gamma_y, \kappa_x, \kappa_y, \theta')^\top$ to $(N, V_x, V_y, M_x, M_y, T)^\top$ whose off-diagonal coupling blocks change whenever the reference point moves.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from numga.topology import Topology
from examples.mechanics.simplex import barycentric_samples, simplex_inertia_weights
from examples.mechanics.bending import render
from examples.mechanics.bending.mesh import section_names, section_walls
from examples.util.animation import save_figure, show_states

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Point = ga.gatype.antivector()
Twist = ga.gatype.bivector()
# Lines of force: loads, and the resultant of everything a section exerts.
Forque = ga.gatype.antibivector()
Motor = ga.gatype.rotor()
Plane = ga.gatype.vector()
# Lines in the section's plane.
Strain = Twist.derive.touching(Algebra("z+"))
# Lines in the section's plane as joins of its points, such as the line through a wall's ends.
WallLine = Strain.derive.flipped_orientation
# Lines along the beam, and couples across it.
Resultant = Strain.derive.dual
# Fibre pulls weighted by signed swept-area warping, and their couples across the beam.
Warped = Resultant.derive.flipped_orientation
Couples = Warped.derive.touching(Algebra("w0"))
# Shifts of a pole across the beam: points at infinity in the section's plane.
Shift = Plane.derive.within(Algebra("x+y+")).derive.dual
# Transverse planes across and up the section, for placing points in `z = 0`.
SectionPlane = Plane.derive.within(Algebra("x+y+"))
# A bending strain to the resultant of the fibres' pulls.
Bending = ga.gatype((Resultant, Strain))                    # Resultant- <- Strain+
# Any strain to the forque of everything the section exerts.
Stiffness = ga.gatype((Forque, Twist))                      # Forque- <- Twist+

# Cross-section plane at the clamp, axial direction, and unit-length unstrained shift twist.
section_plane = mv.z                                        # [] Plane+
beam_direction = section_plane.dual()                       # [] Point-
shift_along_beam = section_plane ^ mv.w                     # [] Twist+
# Pure axial torque couple, the join of the transverse directions across and up.
couple_about_beam = mv.xy.dual()                            # [] Forque-


def section_point(coordinates: np.ndarray) -> Point:
    """Two-dimensional `(x, y)` `coordinates` (`[..., 2]`) embed into unit-weight points (`[...] Point-`)
    in the clamp plane `z = 0`."""
    return (mv(SectionPlane, coordinates) + mv.w).dual()                    # [...] Point-

In [ ]:
# Thin-walled cross-sections represented as one-dimensional edge complexes with node positions and wall thicknesses.
sections = section_walls()
save_figure(render.draw_sections(sections), "bending_cross_sections");

From left to right, the cross-sections are an upright I-section, an I-section turned by thirty degrees, a Z-section, and an open slotted tube.

## Cross-section bending stiffness

Under pure bending and axial extension, adjacent cross-sections remain planar and move relative to one another per unit length of beam by a line in the section plane, `Strain`. When that line crosses the section it acts as the neutral axis about which the section turns, and when it lies on the line at infinity `shift_along_beam` it shifts the entire section uniformly along the beam.

Each point of the cross-section carries an axial fibre `beam_direction & point` whose elongation rate is the regressive product `strain & fibre`. Scaling each fibre by `modulus` times its stretch forms the rank-one dyad `modulus * fibre * (Strain & fibre)`, and integrating those dyads along every straight wall by exact two-point simplex quadrature builds the bending stiffness map from `Strain` to `Resultant`.

In [ ]:
modulus = 1.0

In [ ]:
def wall_samples(walls: Point, thickness: Scalar) -> Point:
    """Two quadrature points along each wall (`[2] Point-[E]`), weighted by the square root of each
    point's share of the wall area, integrate any quadratic over the wall exactly."""
    weights = simplex_inertia_weights(2)
    tail, head = walls                                                      # Point-[E], Point-[E]
    wall_lengths = (tail & head).norm()                                     # Scalar+[E]
    return barycentric_samples(weights, walls) * (wall_lengths * thickness).square_root()


def bending_stiffness(walls: Point, thickness: Scalar, modulus: float) -> Bending:
    """Summing fibre tension dyads over the exact wall quadrature points gives the section's bending
    stiffness (`[] Resultant- <- Strain+`)."""
    fibres = beam_direction & wall_samples(walls, thickness)                # [2] Resultant-[E]
    stretches = Strain & fibres                                             # [2] (Scalar+ <- Strain+)[E]
    # Each sample point appears twice in the dyad, so its square-root area weight contributes once.
    tensions = modulus * fibres * stretches                                 # [2] (Resultant- <- Strain+)[E]
    return tensions.sites.sum().sum(axis=0)                                 # [] Resultant- <- Strain+


z_index = section_names.index("Z-section")
z_topology, z_nodes, z_thickness = sections[z_index]
z_walls = z_topology.incidence[1, 0] * z_nodes                              # [2] Point-[E]
print(bending_stiffness(z_walls, z_thickness, modulus).coefficient_table())

The Z-section's bending stiffness table holds the cross-sectional area, first moments, and second moments of area from a single sum without locating a centroid first. Axial shift `wz` pairs with net axial pull to record the area, while the in-plane turning components `yz` and `zx` pair with transverse bending couples to record the second moments, including the off-diagonal product moment caused by the Z-section's lack of mirror symmetry. First moments vanish here because the Z-section is centred on the origin, and shifting the section off its centroid populates those entries while leaving every solve below unchanged.

## Eccentric axial pushes and the kern

An axial load acting through a point of the cross-section is the resultant line `beam_direction & point`, so `rigidity.solve(beam_direction & point)` maps every load point directly to its neutral axis in `Strain`. Pushing through the centroid compresses every fibre equally and places the neutral axis on the line at infinity, whereas pushing off the centroid tilts the section and brings the neutral axis inward on the opposite side of the centroid.

Where the neutral axis crosses the section, fibres beyond it enter tension. The region of load points that keep the entire cross-section in compression is called the kern. Because `rigidity` is symmetric, a unit pull through one point stretches the fibre through a second point by the same amount as a unit pull through the second stretches the fibre through the first, making `rigidity.solve(beam_direction & point)` a projective polarity between points and lines. Every node's polar line therefore bounds the kern, clipping the plane to the dual polygon of the cross-section's convex hull.

In classical strength-of-materials notation, shifting the origin to the centroid and rotating to principal axes writes the neutral axis of an eccentric load at $(e_x, e_y)$ as $1 + e_x x / i_y^2 + e_y y / i_x^2 = 0$ in terms of the principal radii of gyration $i_x$ and $i_y$.

In [ ]:
axial_push = 1e-3
push_points = section_point(np.array([[0.03, 0.04], [0.1, 0.3]]))           # [2] Point-

In [ ]:
def centroid(walls: Point, thickness: Scalar) -> Point:
    """Area-weighted wall midpoints sum to the normalized centroid (`[] Point-`) of the section."""
    tail, head = walls                                                      # Point-[E], Point-[E]
    wall_lengths = (tail & head).norm()                                     # Scalar+[E]
    return ((tail + head) * (wall_lengths * thickness / 2)).sites.sum().normalized()


def polar_strains(rigidity: Bending, points: Point) -> Strain:
    """Solving `rigidity` against unit axial pulls through `points` maps each point to its polar neutral
    axis (`[...] Strain+`)."""
    return rigidity.solve(beam_direction & points)                          # [...] Strain+


centres = [centroid(topology.incidence[1, 0] * nodes, thickness)
           for topology, nodes, thickness in sections]

In [ ]:
def axial_stress(strains: Twist, modulus: float, points: Point) -> Scalar:
    """Pairing `strains` with the axial fibres through `points` and scaling by `modulus` gives the axial
    normal stress (`[...] Scalar+`)."""
    return modulus * (strains & (beam_direction & points))                  # [...] Scalar+


def axial_pushes(topology: Topology, nodes: Point, thickness: Scalar, points: Point,
                 modulus: float, push: float) -> tuple[Strain, Scalar, Strain]:
    """Compressive axial `push` loads through `points` produce bending strains (`[...] Strain+`), wall-end
    stresses (`[..., 2] Scalar+[E]`), and the node polar lines (`[V] Strain+`) whose envelope bounds the kern."""
    walls = topology.incidence[1, 0] * nodes                                # [2] Point-[E]
    rigidity = bending_stiffness(walls, thickness, modulus)                 # [] Resultant- <- Strain+
    strains = -push * polar_strains(rigidity, points)                       # [...] Strain+
    stresses = axial_stress(strains[..., None], modulus, walls)             # [..., 2] Scalar+[E]
    return strains, stresses, polar_strains(rigidity, nodes.batch())


pushes = [axial_pushes(*section, push_points, modulus, axial_push) for section in sections]
save_figure(render.draw_pushes(sections, centres, push_points, pushes), "bending_kern");

Each panel shades the kern in purple, marks the centroid with a dark dot and the eccentric push point with an orange cross, draws the neutral axis in green, and colours the walls from blue compression to red tension. Pushing inside the kern along the top row keeps the green neutral axis clear of every wall so the entire section stays in blue compression, whereas pushing outside the kern along the bottom row drives the neutral axis across the walls and puts the far side into red tension. Rectangular convex hulls give the two I-sections diamond kerns, the Z-section's slanted hull gives a tilted quadrilateral kern, and the slotted tube's kern sits inside its hollow where an axial load can act through an end plate.

## Warping and the shear centre

When an open thin-walled beam twists about an axial line, its cross-sections warp out of plane. Each midline point displaces axially in proportion to the signed area swept along the walls by the segment from the twisting pole to that point. Twisting about a general pole stretches and compresses the axial fibres with a net bending couple across the beam, whereas at one point called the shear centre those warped fibre pulls exert zero bending couple.

Joining consecutive wall nodes forms the midline segments `tail & head`, whose least-squares integral `(topology.d[0] * WallLine).lstsq(tail & head)` accumulates path lines at the nodes up to a common additive constant corresponding to uniform axial shift. Joining those accumulated lines with `beam_direction` and a `pole` evaluates the signed swept area `beam_direction & pole & paths` directly without coordinates. Because that swept area is linear in `pole`, evaluating `warping` on transverse ideal shifts `Shift` builds a linear map from pole shifts to warping couples, and one linear solve shifts the centroid to the shear centre.

In Vlasov thin-walled beam notation, warping is written as the sectorial coordinate $\omega(s) = \int_0^s r_\perp\,ds$ about a chosen pole, and locating the shear centre solves two coupled area integrals so that $\int \omega x\,dA = 0$ and $\int \omega y\,dA = 0$.

In [ ]:
def warping(topology: Topology, nodes: Point, thickness: Scalar, pole: Point) -> Warped:
    """Integrating fibres weighted by the signed area swept from `pole` along the walls (`[E]`) gives
    the warping resultant (`[] Warped+`)."""
    walls = topology.incidence[1, 0] * nodes                                # [2] Point-[E]
    tail, head = walls                                                      # Point-[E], Point-[E]
    # Least-squares integration of wall lines along `d[0]` accumulates path lines at the nodes up to a common constant.
    paths = (topology.d[0] * WallLine).lstsq(tail & head)                   # WallLine-[V]
    # Joining the accumulated wall lines with `pole` and `beam_direction` measures the signed swept area.
    node_warpings = beam_direction & pole & paths                           # Scalar-[V]
    wall_lengths = (tail & head).norm()                                     # Scalar+[E]
    area_roots = (wall_lengths * thickness).square_root()                   # Scalar+[E]
    end_warpings = topology.incidence[1, 0] * node_warpings                 # [2] Scalar-[E]
    sample_warpings = barycentric_samples(simplex_inertia_weights(2), end_warpings) * area_roots  # [2] Scalar-[E]
    return (sample_warpings * (beam_direction & wall_samples(walls, thickness))).sites.sum().sum(axis=0)  # [] Warped+


def shear_centre(topology: Topology, nodes: Point, thickness: Scalar) -> Point:
    """Shifting the pole from the centroid cancels the centroid's warping couples across the beam to
    locate the shear centre (`[] Point-`)."""
    centre = centroid(topology.incidence[1, 0] * nodes, thickness)          # [] Point-
    couples_per_shift = warping(topology, nodes, thickness, Shift).cast(Couples)  # [] Couples+ <- Shift-
    centroid_couples = warping(topology, nodes, thickness, centre).cast(Couples)  # [] Couples+
    return centre - couples_per_shift.solve(centroid_couples)               # [] Point-


shear_centres = [shear_centre(*section) for section in sections]
save_figure(render.draw_walls(sections, centres, shear_centres), "bending_walls");

Each cross-section shows its centroid as a solid dot and its shear centre as an open ring. Half-turn symmetry about the centroid places the shear centre directly on the centroid for both I-sections and for the Z-section, whereas the slotted tube's shear centre lies outside the wall on the side opposite the opening.

## Full section stiffness

Relative motion between neighbouring cross-sections spans all six components of `Twist`, combining axial extension, two bending turns, axial torsion, and two transverse shear slides. Open walls resist Saint-Venant torsion by `shear_modulus` times a third of each wall's length times the cube of its thickness, acting along the pure axial torque `couple_about_beam`. Timoshenko shear adds each wall's resistance to transverse sliding along its own midline in proportion to `shear_modulus` times wall area.

Joining the shear centre with each unit wall direction forms the shear line `slide_lines`, and summing its dyads alongside the fibre tension dyads and torsional dyad builds `section_stiffness` over the full `Twist` space. Every wall's shear line passes through the shear centre, so an axial twist line through the shear centre meets all of them with zero regressive pairing `slide_lines & Twist` and incurs only torsional resistance, whereas twisting about any other axial line also slides the walls. Without prescribing a twisting axis in advance, solving against `section_stiffness` places pure twist automatically through the shear centre.

In [ ]:
poisson_ratio = 0.3
shear_modulus = modulus / (2 * (1 + poisson_ratio))

In [ ]:
def torsional_stiffness(walls: Point, thickness: Scalar, shear_modulus: float) -> Scalar:
    """Open walls resist Saint-Venant torsion by `shear_modulus` times one third of each wall's length
    times the cube of its thickness (`[] Scalar+`)."""
    tail, head = walls                                                      # Point-[E], Point-[E]
    wall_lengths = (tail & head).norm()                                     # Scalar+[E]
    return shear_modulus * (wall_lengths * thickness**3 / 3).sites.sum()    # [] Scalar+


def section_stiffness(topology: Topology, nodes: Point, thickness: Scalar, modulus: float,
                      shear_modulus: float) -> Stiffness:
    """Fibre tension, Saint-Venant torsion, and wall shear through the shear centre sum as line dyads
    into the full section stiffness (`[] Forque- <- Twist+`)."""
    walls = topology.incidence[1, 0] * nodes                                # [2] Point-[E]
    tail, head = walls                                                      # Point-[E], Point-[E]
    fibres = beam_direction & wall_samples(walls, thickness)                # [2] Resultant-[E]
    axial_and_bending = (modulus * fibres * (Twist & fibres)).sites.sum().sum(axis=0)  # [] Resultant- <- Twist+
    torsion_constant = torsional_stiffness(walls, thickness, shear_modulus)  # [] Scalar+
    torsion = torsion_constant * couple_about_beam * (couple_about_beam & Twist)  # [] Forque- <- Twist+
    wall_lengths = (tail & head).norm()                                     # Scalar+[E]
    wall_directions = (topology.d[0] * nodes) / wall_lengths                # Point-[E]
    slide_lines = shear_centre(topology, nodes, thickness) & wall_directions  # Forque-[E]
    shear = (shear_modulus * wall_lengths * thickness * slide_lines * (slide_lines & Twist)).sites.sum()  # [] Forque- <- Twist+
    return axial_and_bending + torsion + shear                              # [] Forque- <- Twist+

## Cantilevers under a tip load

Clamping each beam at `z = 0` and applying a downward load through the tip centroid subjects every cross-section station to the resultant of the tip load seen in that station's frame. Translating the tip load back toward the clamp by `shifts << loads[..., None]` expresses the internal shear force, bending couple, and torsional torque about the station in a single `Forque` line, so `stiffness.solve(-station_loads(loads, stations))` recovers all six twist strain components at every station at once.

Adding `shift_along_beam` to each piece strain and exponentiating `((shift_along_beam + strains) * (step / 2)).exp()` integrates constant piece twists into rigid motors that step from section to section along the cantilever. Station lever arms are evaluated on the straight reference beam under small deflections with free warping at the clamp.

In [ ]:
beam_length = 10.0
n_pieces = 40
tip_load = 1e-5
up_direction = mv.y.dual()                                                  # [] Point-
down_load = tip_load * -up_direction                                        # [] Point-

In [ ]:
def station_loads(loads: Forque, stations: np.ndarray) -> Forque:
    """Translating tip `loads` back by `stations` (`[n_pieces]`) expresses the internal load in each
    station's cross-section plane (`[..., n_pieces] Forque-`)."""
    shifts = (shift_along_beam * (stations / 2)).exp()                      # [n_pieces] Motor+
    return shifts << loads[..., None]                                       # [..., n_pieces] Forque-


def section_frames(strains: Twist, step: float) -> Motor:
    """Exponentiating each piece twist (`[..., n_pieces] Twist+`) over `step` and multiplying rightwards
    from the clamp builds the cross-section frames (`[..., n_pieces + 1] Motor+`)."""
    piece_motors = ((shift_along_beam + strains) * (step / 2)).exp()        # [..., n_pieces] Motor+
    clamp = mv.rotor().broadcast_to(piece_motors.shape[:-1] + (1,))         # [..., 1] Motor+
    # Each piece twist acts in the local frame of the preceding section, so cumulative right-multiplication
    # equals the reverse of the cumulative left-product of reversed motors.
    return concatenate([clamp, piece_motors.reverse().cumprod(axis=-1).reverse()], axis=-1)  # [..., n_pieces + 1] Motor+

In [ ]:
def tip_points(points: Point, length: float) -> Point:
    """Translating clamp-plane `points` by `length` along `shift_along_beam` places them on the unbent
    tip section (`[...] Point-`)."""
    shift = (shift_along_beam * (length / 2)).exp()                         # [] Motor+
    return shift >> points                                                  # [...] Point-


def cantilever(topology: Topology, nodes: Point, thickness: Scalar, loads: Forque, length: float,
               modulus: float, shear_modulus: float, n_pieces: int) -> tuple[Twist, Motor, Strain, Scalar]:
    """Solving the section stiffness along `n_pieces` stations of a cantilever of `length` under tip
    `loads` gives the piece strains (`[..., n_pieces] Twist+`), section frames (`[..., n_pieces + 1] Motor+`),
    two principal bending axes (`[2] Strain+`), and clamp wall-end stresses (`[..., 2] Scalar+[E]`)."""
    step = length / n_pieces
    stations = (np.arange(n_pieces) + 0.5) * step                           # [n_pieces]
    walls = topology.incidence[1, 0] * nodes                                # [2] Point-[E]
    rigidity = bending_stiffness(walls, thickness, modulus)                 # [] Resultant- <- Strain+
    stiffness = section_stiffness(topology, nodes, thickness, modulus, shear_modulus)  # [] Forque- <- Twist+
    # Station lever arms are evaluated on the straight reference beam; `exp(strain * step / 2)` moves frames
    # opposite to the bivector's own sign, so the stiffness solves against `-station_loads(...)`.
    strains = stiffness.solve(-station_loads(loads, stations))              # [..., n_pieces] Twist+
    frames = section_frames(strains, step)                                  # [..., n_pieces + 1] Motor+
    # The Euclidean metric on `Strain` vanishes on pure axial shift, ordering the two bending axes before it.
    _, axes = (Strain | Strain).eigh(Strain & rigidity)                     # [3] Strain+
    clamp_stresses = axial_stress(strains[..., 0, None], modulus, walls)    # [..., 2] Scalar+[E]
    return strains, frames, axes[:2], clamp_stresses


beams = [cantilever(*section, tip_points(centre, beam_length) & down_load,
                    beam_length, modulus, shear_modulus, n_pieces)
         for section, centre in zip(sections, centres)]
save_figure(render.draw_beams(sections, centres, shear_centres, down_load, beams), "bending_sections");

Along the top row, each clamp section is coloured by axial stress from blue compression to red tension and overlaid with the deflected tip section in solid blue, the orange tip load arrow, the green neutral axis, and the two dashed grey principal bending axes through the centroid; along the bottom row, each bent cantilever appears beside its translucent straight reference beam with the locus of shear centres dashed in dark grey. Mirror symmetry keeps the upright I-section's neutral axis horizontal so its tip deflects straight down, whereas both the turned I-section and the Z-section tilt their neutral axes toward their weaker principal bending axis and deflect sideways as well as downward. Loading the slotted tube through its centroid misses the exterior shear centre ring, so the tube twists substantially as it bends.

In matrix Euler–Bernoulli and Saint-Venant notation, bending curvatures about centroidal axes solve $\begin{pmatrix}\kappa_x \\ \kappa_y\end{pmatrix} = \frac{1}{E}\begin{pmatrix}I_{yy} & -I_{xy} \\ -I_{xy} & I_{xx}\end{pmatrix}^{-1}\begin{pmatrix}M_x \\ M_y\end{pmatrix}$, torsion solves $\theta' = T / (G J)$ with $J = \sum b t^3 / 3$ and torque $T$ shifted to the shear centre, and transverse shear solves $\gamma = V / (G A_s)$.

## Rotating tip load on the Z-section

Rotating the tip load through a full turn in the Z-section's plane sweeps the neutral axis around the centroid while keeping the load line through the shear centre at every angle, so the cantilever never twists. Whenever the orange load aligns with either dashed principal axis, the blue tip section deflects straight along the load; at intermediate angles it swings toward the weaker principal axis and traces an ellipse aligned with the principal bending directions.

In [ ]:
n_frames = 72
frame_duration_ms = 40

In [ ]:
def turning_states(topology: Topology, nodes: Point, thickness: Scalar, centre: Point,
                   down_load: Point, length: float, modulus: float, shear_modulus: float,
                   n_pieces: int, n_frames: int
                   ) -> Iterator[tuple[Point, Point, Twist, Motor, Strain, Scalar]]:
    """Yields `(load_point, load_direction, strains, frames, principal_axes, clamp_stresses)` across
    `n_frames` as the tip load rotates one full turn in the cross-section plane."""
    angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)            # [n_frames]
    tip_centre = tip_points(centre, length)                                 # [] Point-
    for angle in angles:
        turn = (mv.xy * (-angle / 2)).exp()                                 # [] Motor+
        load_direction = turn >> down_load                                  # [] Point-
        strains, frames, axes, stresses = cantilever(
            topology, nodes, thickness, tip_centre & load_direction,
            length, modulus, shear_modulus, n_pieces)
        yield centre, load_direction, strains, frames, axes, stresses


turning = turning_states(*sections[z_index], centres[z_index], down_load,
                         beam_length, modulus, shear_modulus, n_pieces, n_frames)
turned_states = show_states(
    turning, render.animate_loads, sections[z_index], centres[z_index], shear_centres[z_index],
    name="bending_turning", duration_ms=frame_duration_ms, n_frames=n_frames)

## Translating tip load across the slotted tube

Translating a downward tip load horizontally across the slotted tube's symmetry line from the open side past the exterior shear centre holds the vertical bending couple constant while varying the torque arm about the shear centre. As the load approaches the shear centre ring, the twist shrinks to zero, and once the load passes beyond the ring the tube twists in the opposite sense while maintaining the same vertical deflection.

In [ ]:
start_point, end_point = section_point(np.array([[0.4, 0.0], [-1.15, 0.0]]))  # [] Point-, [] Point-

In [ ]:
def crossing_states(topology: Topology, nodes: Point, thickness: Scalar, start: Point, end: Point,
                    down_load: Point, length: float, modulus: float, shear_modulus: float,
                    n_pieces: int, n_frames: int
                    ) -> Iterator[tuple[Point, Point, Twist, Motor, Strain, Scalar]]:
    """Yields `(load_point, load_direction, strains, frames, principal_axes, clamp_stresses)` across
    `n_frames` as a downward tip load translates linearly from `start` to `end` across the section."""
    fractions = np.linspace(0, 1, n_frames)                                 # [n_frames]
    for fraction in fractions:
        load_point = start * (1 - fraction) + end * fraction                # [] Point-
        strains, frames, axes, stresses = cantilever(
            topology, nodes, thickness, tip_points(load_point, length) & down_load,
            length, modulus, shear_modulus, n_pieces)
        yield load_point, down_load, strains, frames, axes, stresses


tube_index = section_names.index("slotted tube")
crossing = crossing_states(*sections[tube_index], start_point, end_point, down_load,
                           beam_length, modulus, shear_modulus, n_pieces, n_frames)
crossed_states = show_states(
    crossing, render.animate_loads, sections[tube_index], centres[tube_index], shear_centres[tube_index],
    name="bending_across", duration_ms=frame_duration_ms, n_frames=n_frames)

In [ ]:
# checks
for push_strains, push_stresses, node_polars in pushes:
    # Pushing inside the kern compresses every wall end; pushing outside puts some wall end in tension.
    assert (push_stresses[0] < 0).all() and (push_stresses[1] > 0).any()
    # Betti reciprocity holds on the two push strains already solved: each stretches the other's fibre equally.
    reciprocity_gap = ((push_strains[0] & (beam_direction & push_points[1]))
                       - (push_strains[1] & (beam_direction & push_points[0])))  # [] Scalar+
    assert reciprocity_gap.abs() < 1e-12

for centre, shear in zip(centres[:3], shear_centres[:3]):
    assert (shear & centre).norm() < 1e-12
assert bool((mv.x & shear_centres[tube_index]) / (mv.w & shear_centres[tube_index])
            < (mv.x & centres[tube_index]) / (mv.w & centres[tube_index]))

displaced_centres = stack([frames[-1] >> centre for centre, (_, frames, _, _) in zip(centres, beams)])  # [4] Point-
rest_centres = stack(centres)                                               # [4] Point-
sideways = (mv.x & displaced_centres) / (mv.w & displaced_centres) - (mv.x & rest_centres) / (mv.w & rest_centres)  # [4] Scalar+
falls = (mv.y & displaced_centres) / (mv.w & displaced_centres) - (mv.y & rest_centres) / (mv.w & rest_centres)  # [4] Scalar+
assert (falls < 0).all() and bool(sideways[0].abs() < 1e-12)
assert sideways[1:3].abs().min() > 0.5 * falls[1:3].abs().max()

# Rotating the load through the Z-section's centroid and shear centre bends the cantilever with zero twist.
turned_strains = stack([state[2][0] for state in turned_states])            # [n_frames] Twist+
assert ((turned_strains & shift_along_beam).abs() < 1e-12).all()

# Translating the downward load across the slotted tube reverses the twist across the shear centre.
crossed_points = stack([state[0] for state in crossed_states])              # [n_frames] Point-
crossed_strains = stack([state[2][0] for state in crossed_states])          # [n_frames] Twist+
twist_rates = crossed_strains & shift_along_beam                            # [n_frames] Scalar+
assert twist_rates[0] * twist_rates[-1] < 0
shear_offsets = ((mv.x & crossed_points) / (mv.w & crossed_points)
                 - (mv.x & shear_centres[tube_index]) / (mv.w & shear_centres[tube_index])).abs()  # [n_frames] Scalar+
passing_frame = int(shear_offsets.argmin())
assert twist_rates[passing_frame].abs() < 0.1 * twist_rates.abs().max()